# Pharmacy Medicine Demand & Expiry-Risk Forecasting
## Algions AI Labs Private Limited - AI/ML Assessment
### Exploratory Data Analysis & Modeling Pipeline

**Author:** AI/ML Engineering Candidate  
**Domain:** Pharmacy Supply Chain Decision-Support  
**Disclaimer:** All data is synthetic; outputs are advisory operational guidance only.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import load_sales_data, load_inventory_data, clean_sales_data, clean_inventory_data
from src.eda import run_full_eda

sns.set_theme(style='whitegrid')
%matplotlib inline


### 1. Load and Clean Datasets

In [ ]:
sales_df = load_sales_data('../dataset/daily_medicine_sales.csv')
inv_df = load_inventory_data('../dataset/batch_expiry_inventory.csv')

sales_clean = clean_sales_data(sales_df)
inv_clean = clean_inventory_data(inv_df)

print(f'Sales shape: {sales_clean.shape}')
print(f'Inventory shape: {inv_clean.shape}')


### 2. Dataset Overview & Key Statistics

In [ ]:
eda_results = run_full_eda(sales_clean, inv_clean, output_dir='../outputs/eda_visualizations')
print('Sales Overview:')
for k, v in eda_results['sales_overview'].items():
    print(f'  {k}: {v}')

print('
Inventory Overview:')
for k, v in eda_results['inventory_overview'].items():
    print(f'  {k}: {v}')


### 3. Expiry-Risk Heuristic Analysis
Demonstrating the decision boundary: days_to_expiry <= 90 & projected_unsold_ratio >= 0.30


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=inv_clean,
    x='days_to_expiry',
    y='projected_unsold_ratio',
    hue='expiry_risk',
    palette={'Low': '#5cb85c', 'Medium': '#f0ad4e', 'High': '#d9534f'},
    alpha=0.7
)
plt.axvline(90, color='#d9534f', linestyle='--', label='High Risk Cutoff (90d)')
plt.axvline(180, color='#f0ad4e', linestyle='--', label='Medium Risk Cutoff (180d)')
plt.axhline(0.30, color='#d9534f', linestyle=':', label='High Ratio Cutoff (30%)')
plt.axhline(0.15, color='#f0ad4e', linestyle=':', label='Medium Ratio Cutoff (15%)')
plt.title('Batch Expiry Risk Decision Boundary')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### 4. Demand Trend and Stock Analysis

In [ ]:
daily_summary = sales_clean.groupby('date').agg({'units_sold': 'sum', 'stock_on_hand': 'sum'}).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(daily_summary['date'], daily_summary['units_sold'], color='#d9534f', label='Total Units Sold')
ax1.set_ylabel('Daily Units Sold', color='#d9534f')
ax1.set_title('Daily Total Sales and Stock Across All 50 Medicines')
plt.show()
